# 单文件 LFP 流程

本 Notebook 可在真实 FIF 到位时运行真实文件；若路径不存在，则自动切换到明确标记的合成信号，仅用于算法验证。它不补造动物身份、给药信息或 AIMs。

In [ ]:
from pathlib import Path
from datetime import UTC, datetime
import os
import uuid
import sys
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from lfp_analysis.pipeline import run_single_file
from lfp_analysis.synthetic import validate_synthetic

# Set LUNA_NOTEBOOK_INPUT to a FIF path for a real-file run. Leaving it empty
# deliberately selects the synthetic fallback; no local machine path is embedded.
input_text = os.environ.get('LUNA_NOTEBOOK_INPUT', '').strip()
sample_path = Path(input_text).expanduser() if input_text else None
output_text = os.environ.get('LUNA_NOTEBOOK_OUTPUT_DIR', '').strip()
run_output = Path(output_text).expanduser() if output_text else PROJECT_ROOT / 'results' / f'notebook_{datetime.now(UTC).strftime("%Y%m%d_%H%M%S")}_{uuid.uuid4().hex[:8]}'
run_output = run_output.resolve()
if run_output.exists() and any(run_output.iterdir()):
    raise FileExistsError(f'Refusing to overwrite non-empty notebook output: {run_output}')
real_output = run_output  # existing display cells use this name for either mode
if sample_path is not None and sample_path.is_file():
    manifest = run_single_file(sample_path, PROJECT_ROOT / 'configs' / 'luna.yaml', run_output, PROJECT_ROOT / 'metadata')
    run_mode = 'REAL FIF (local read-only input)'
else:
    manifest = validate_synthetic(run_output)
    run_mode = 'SYNTHETIC VALIDATION ONLY'
print(run_mode)
manifest

In [ ]:
if 'REAL' in run_mode:
    quality_file = pd.read_csv(real_output / 'quality_file.csv')
    quality_channel = pd.read_csv(real_output / 'quality_channel.csv')
    epochs_trace = pd.read_csv(real_output / 'epochs_trace.csv')
    display(quality_file)
    display(quality_channel.loc[quality_channel['n_warn_epochs'] > 0])
    display(epochs_trace.loc[epochs_trace['drop_reason'].notna() & epochs_trace['drop_reason'].ne('')])
    parameterization_model = pd.read_csv(real_output / 'parameterization_model.csv')
    parameterization_peaks = pd.read_csv(real_output / 'parameterization_peaks.csv')
    display(parameterization_model[['channel_name', 'backend_used', 'fit_status', 'r_squared', 'offset', 'exponent']])
    display(parameterization_peaks.groupby('channel_name', as_index=False).size().rename(columns={'size': 'n_peaks'}))
else:
    display(manifest)

## 解释边界

T80 是名义给药后时点；本流程不把 epoch 的 events 值解释成原始记录起止时间。没有登记动物身份、给药天数和行为同步时，Notebook 只展示文件级结果。

## 功能连接：MIC、MIM 与去偏平方 wPLI

连接估计在同一文件/记录节点/给药时点内跨多个有效 epoch 完成。MIC/MIM 使用四通道脑区集合的多变量估计；wPLI 保留全部跨脑区通道对。当前样例没有可用于动物层推断的身份信息。

In [ ]:
if 'REAL' in run_mode:
    connectivity_rank = pd.read_csv(real_output / 'connectivity_rank_summary.csv')
    connectivity_checks = pd.read_csv(real_output / 'connectivity_input_checks.csv')
    connectivity_bands = pd.read_csv(real_output / 'connectivity_band_summary.csv')
    connectivity_spectrum = pd.read_csv(real_output / 'connectivity_spectrum.csv')
    print('Connectivity status:', manifest['connectivity_status'])
    print('Epochs:', manifest['n_epochs'], '| effective duration (s):', manifest['effective_valid_duration_s'])
    print('Spectrum rows:', len(connectivity_spectrum), '| band rows:', len(connectivity_bands))
    display(connectivity_rank[['region', 'n_channels', 'numerical_rank', 'variance_rank', 'selected_rank', 'covariance_condition_number']])
    display(connectivity_checks)
    display(connectivity_bands[['method', 'region_a', 'region_b', 'band', 'value_raw_or_summary', 'value_strength', 'n_epochs', 'rank_seed', 'rank_target', 'status']].head(24))
else:
    print('Connectivity is not run in synthetic fallback mode.')
    display(manifest)

### 连接结果的当前限制

本次只支持单文件描述性连接结果；等量条件抽样在单文件上不适用，Granger/时间反转校正默认关闭。MIM 保留未归一化原值，wPLI 不开平方且不把负估计截为零。不要把通道对、epoch 或保留维度当成动物样本。

## 时间延迟：PyBispectra TDE

TDE 在同一文件内跨有效 epoch 估计，不拼接不连续片段。标准 Method I 和 bispectral antisymmetrized 结果同时保留；正值仅表示 seed→target 的时间符号约定，不表示解剖因果方向。

In [ ]:
if 'REAL' in run_mode:
    tde_meta = json.loads((real_output / 'time_delay_metadata.json').read_text(encoding='utf-8'))
    tde_checks = pd.read_csv(real_output / 'time_delay_input_checks.csv')
    tde_band = pd.read_csv(real_output / 'time_delay_band_summary.csv')
    tde_pair = pd.read_csv(real_output / 'time_delay_channel_pair_summary.csv')
    print('TDE status:', manifest['time_delay_status'])
    print('Analysis sampling rate (Hz):', tde_meta['analysis_sfreq_hz'], '| delay window (ms):', tde_meta['delay_window_ms'], '| delay resolution (ms):', tde_meta['delay_resolution_ms'])
    print('TDE spectrum rows:', len(pd.read_csv(real_output / 'time_delay_spectrum.csv')))
    display(tde_checks)
    display(tde_band[['region_a', 'region_b', 'antisymmetrized', 'frequency_band', 'region_peak_delay_ms', 'channel_pair_median_delay_ms', 'channel_pair_mad_delay_ms', 'quality_flag']].head(24))
    display(tde_pair.groupby(['antisymmetrized', 'direction_relative_to_seed_target'], dropna=False).size().reset_index(name='n_channel_pair_results'))
else:
    print('TDE is not run in synthetic fallback mode.')
    display(manifest)

### TDE 当前限制

TDE 的延迟窗口、降采样和频段均为配置化起步设置。峰贴近窗口边界、跨通道对离散较大或区域峰与通道对中位数不一致时，只能作为低稳定性结果保留，不能直接进入动物层统计。